# Phase 5 submission — the confirmed 2-series model

`phase5_confirm_efficientnet_b0_s2x16_e8_onecycle_amp_lat1_aug1`, promoted on the pre-registered
rule: pooled OOF **0.8523** vs run 1's 0.7779, paired delta **+0.0744, 95% CI [+0.0669, +0.0820]**.
Gold transfer **0.8189** against run 1's 0.7252 — and run 1's 0.7252 preceded an LB of 0.763.

Changes from `phase4-submit`, all of them consequences of what the model was trained on:

- **`MAX_SERIES = 2`.** The read config must match training or the model is fed an input shape it
  never saw. Prep parameters stay at Phase 2's (`max_series=4, k_slices=24, size=256`) for the
  reason recorded in NOTES 2026-09-02: `select_series` exhausts the priority list and then falls
  back to arbitrary dataframe order, so prepping fewer series can select *different* series.
- **Phase 5 checkpoints**, five folds, mean of the sigmoids.
- **No augmentation.** `PreppedStudyDataset` defaults `augment=False`; train-time jitter at
  inference would just add variance.

**The failure this kernel guards against.** Training read corrected laterality via the `sides`
override; test studies instead get their side from `prep_study`'s freshly written meta, which comes
from `resolve_study_laterality` — and that only agrees with training if the mounted src carries the
geometry fallback. A stale src would silently leave ~49% of test studies unmirrored while the model
was trained on mirrored ones, on four medial/lateral labels, with no error anywhere. So the import
is asserted up front and the resolved-route distribution is printed and checked.

In [ ]:
import glob, os, shutil, sys, tempfile, time

GIT_SHA = 'phase5-submit'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
CKPTS = sorted(glob.glob('/kaggle/input/**/knee_phase5_fold*.pt', recursive=True))
assert len(CKPTS) == 5, f'expected 5 fold checkpoints, found {len(CKPTS)}: {CKPTS}'

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

# A stale src dataset would silently un-mirror ~49% of test studies (see header).
from knee.dicom import side_from_geometry, resolve_study_laterality
import inspect
assert callable(side_from_geometry), 'src predates the geometry laterality route'
assert 'side_from_geometry' in inspect.getsource(resolve_study_laterality), \
    'src resolve_study_laterality has no geometry fallback -- test studies would not match training'
print('src carries the geometry laterality fallback')

print('git sha      ', GIT_SHA)
print('src          ', SRC)
print('competition  ', COMP_DIR)
for p in CKPTS:
    print('checkpoint   ', p)

In [ ]:
import numpy as np
import pandas as pd
import torch

from knee.infer import LABEL_COLUMNS, build_submission
from knee.dataset import PreppedStudyDataset
from knee.model import KneeModel
from knee.prep import prep_study, save_study_npz

# Phase 2's prep parameters and run 1's read parameters -- see the header cell.
PREP_K, PREP_SIZE, PREP_MAX_SERIES = 24, 256, 4
MAX_SERIES, N_SLICES = 2, 16   # matches the confirmed config

test_df = pd.read_csv(f'{COMP_DIR}/test.csv')
test_series_df = pd.read_csv(f'{COMP_DIR}/test_series.csv')
TEST_ROOT = f'{COMP_DIR}/test_series'
print(f'{len(test_df)} test studies')

# Probe rather than assert: this is the scored submission, and a hard GPU
# requirement turns an odd accelerator assignment into a zero.
device = 'cpu'
if torch.cuda.is_available():
    try:
        _ = torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except Exception as e:
        print(f'GPU present but unusable with this torch build: {e}')
print('device:', device)

# pretrained=False: internet is off, and every weight is about to be overwritten
# by the checkpoint anyway. weights_only=True -- our own state_dicts are plain
# tensors, so there is no reason to allow arbitrary unpickling.
models = []
for path in CKPTS:
    m = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS), pretrained=False)
    m.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    models.append(m.to(device).eval())
print(f'{len(models)} fold models loaded')

In [ ]:
NPZ_DIR = tempfile.mkdtemp(prefix='knee_prep_')
prep_seconds, forward_seconds = [], []
lat_routes = {}

def predict_one(study_uid):
    """prep -> npz -> the tested loader -> mean of 5 fold models. Anything that
    raises in here becomes a row of 0.5 via build_submission's fallback."""
    t0 = time.time()
    series_slices, meta = prep_study(
        study_uid, TEST_ROOT, test_series_df,
        k_slices=PREP_K, size=PREP_SIZE, max_series=PREP_MAX_SERIES)
    npz_path = os.path.join(NPZ_DIR, f'{study_uid}.npz')
    save_study_npz(npz_path, series_slices, meta)
    prep_seconds.append(time.time() - t0)
    # what actually decided this study's mirroring, counted for the check below
    lat_routes[meta.get('route')] = lat_routes.get(meta.get('route'), 0) + 1

    try:
        dataset = PreppedStudyDataset([study_uid], NPZ_DIR,
                                      n_slices=N_SLICES, max_series=MAX_SERIES)
        image, _, _ = dataset[0]
        t1 = time.time()
        batch = image.unsqueeze(0).to(device)
        with torch.no_grad():
            probs = np.mean([torch.sigmoid(m(batch))[0].cpu().numpy() for m in models], axis=0)
        forward_seconds.append(time.time() - t1)
    finally:
        # keep peak disk at one study: the test set size is not known in advance
        os.remove(npz_path)
    return probs

wall_start = time.time()
submission = build_submission(test_df['StudyInstanceUID'].tolist(), predict_one)
wall_total = time.time() - wall_start

n_fallback = len(test_df) - len(prep_seconds)
print(f'total wall time for {len(test_df)} studies: {wall_total:.1f}s ({wall_total/60:.1f} min)')
if prep_seconds:
    print(f'prep    {np.mean(prep_seconds):.3f}s/study (p95 {np.percentile(prep_seconds, 95):.3f}s)')
if forward_seconds:
    print(f'forward {np.mean(forward_seconds):.3f}s/study for {len(models)} models')
print(f'studies that hit the 0.5 fallback: {n_fallback}')
print('laterality routes on the test set:', lat_routes)
_geom = sum(v for k, v in lat_routes.items() if str(k).startswith('geometry'))
assert 'unknown' not in lat_routes or _geom > 0, (
    'no test study resolved by geometry while some are unknown -- the fallback is not running, '
    'and training/inference disagree on mirroring')

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission.head())

assert len(submission) == len(test_df), 'row count must match test studies exactly'
assert list(submission.columns) == ['StudyInstanceUID'] + LABEL_COLUMNS, 'column contract'
assert submission[LABEL_COLUMNS].isna().sum().sum() == 0, 'no NaNs allowed'
vals = submission[LABEL_COLUMNS].to_numpy()
assert (vals >= 0).all() and (vals <= 1).all(), 'all predictions must be in [0, 1]'
print('submission.csv passed shape/range checks')

# Cheap distribution sanity check: the model was trained against these positive
# rates, so predictions wildly off them mean something broke upstream of scoring
# (a preprocessing mismatch shows up here before it shows up on the LB).
TRAIN_POSITIVE_RATE = {
    'ACL': 0.0819, 'MCL': 0.0204, 'Medial Meniscus': 0.3186, 'Lateral Meniscus': 0.1037,
    'Medial OA': 0.0740, 'Lateral OA': 0.0245, 'PF OA': 0.0579, 'Effusion': 0.2616,
    'Synovitis': 0.1125, "Baker's": 0.1470, 'Contusion': 0.1028, 'Fracture': 0.0138,
}
print(f'\n{n_fallback} of {len(test_df)} studies fell back to 0.5 -- at a high fallback '
      'count the means below are dominated by that constant, not by the model')
print(f'{"label":20s} {"mean pred":>10s} {"train rate":>11s}')
for label in LABEL_COLUMNS:
    print(f'{label:20s} {submission[label].mean():10.4f} {TRAIN_POSITIVE_RATE[label]:11.4f}')